# 03. Deep Learning for Survival Analysis with `pycox`

## Introduction
Deep Learning (DL) models allow us to learn highly complex, non-linear representations of covariates directly from data. 

We will use the **`pycox`** library, which is built on top of **PyTorch**.

### Models Covered
1. **DeepSurv**: A Cox Proportional Hazards model with a Neural Network replacing the linear combination of features.
   $$h(t|x) = h_0(t) \exp(g_\theta(x))$$
   where $g_\theta(x)$ is a neural network.

In [ ]:
# Install pycox and torch
!pip install pycox torch torchtuples pandas matplotlib scikit-learn

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torchtuples as tt # Helper for pycox

from pycox.datasets import metabric
from pycox.models import CoxPH
from pycox.evaluation import EvalSurv
from sklearn.preprocessing import StandardScaler

np.random.seed(42)
_ = torch.manual_seed(42)

## 1. Load Data
We will use the **METABRIC** breast cancer dataset.
- `duration`: Survival time.
- `event`: Censoring status (1=Event, 0=Censored).
- Gene expression and clinical features.

In [ ]:
df_train = metabric.read_df()
df_test = df_train.sample(frac=0.2)
df_train = df_train.drop(df_test.index)
df_val = df_train.sample(frac=0.2)
df_train = df_train.drop(df_val.index)

print(f"Train: {df_train.shape}, Val: {df_val.shape}, Test: {df_test.shape}")

## 2. Preprocessing
Neural Networks require standardized inputs.

In [ ]:
cols_standardize = ['x0', 'x1', 'x2', 'x3', 'x8']
cols_leave = ['x4', 'x5', 'x6', 'x7']

scaler = StandardScaler()
df_train[cols_standardize] = scaler.fit_transform(df_train[cols_standardize])
df_val[cols_standardize] = scaler.transform(df_val[cols_standardize])
df_test[cols_standardize] = scaler.transform(df_test[cols_standardize])

# Convert to float32 for PyTorch
x_train = df_train.drop(['duration', 'event'], axis=1).values.astype('float32')
x_val = df_val.drop(['duration', 'event'], axis=1).values.astype('float32')
x_test = df_test.drop(['duration', 'event'], axis=1).values.astype('float32')

y_train = (df_train['duration'].values.astype('float32'), df_train['event'].values.astype('float32'))
y_val = (df_val['duration'].values.astype('float32'), df_val['event'].values.astype('float32'))
y_test = (df_test['duration'].values.astype('float32'), df_test['event'].values.astype('float32'))

val_data = tt.tuplefy(x_val, y_val)

## 3. Define the Neural Network
We create a simple Multi-Layer Perceptron (MLP).

In [ ]:
in_features = x_train.shape[1]
num_nodes = [32, 32]
out_features = 1 # DeepSurv outputs a single risk score
batch_norm = True
dropout = 0.1
output_bias = False

net = tt.practical.MLPVanilla(in_features, num_nodes, out_features, batch_norm,
                              dropout, output_bias=output_bias)

## 4. Train DeepSurv
We use the `CoxPH` model class from `pycox`.

In [ ]:
model = CoxPH(net, tt.optim.Adam)
batch_size = 256
lr = 0.01
epochs = 100
verbose = True

model.optimizer.set_lr(lr)

log = model.fit(x_train, y_train, batch_size, epochs, callbacks=[tt.callbacks.EarlyStopping()],
                val_data=val_data, verbose=verbose)

In [ ]:
_ = log.plot()

## 5. Evaluation
We evaluate using the Concordance Index.

In [ ]:
_ = model.compute_baseline_hazards()
surv = model.predict_surv_df(x_test)

ev = EvalSurv(surv, df_test['duration'], df_test['event'], censor_surv='km')
print(f"C-index: {ev.concordance_td():.3f}")

### Integrated Brier Score

In [ ]:
time_grid = np.linspace(df_test['duration'].min(), df_test['duration'].max(), 100)
print(f"Integrated Brier Score: {ev.integrated_brier_score(time_grid):.3f}")